# Exercise 5

In [ ]:
from functions import RandomnessTests
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import random
random.seed(42)
import time
import tracemalloc
np.random.seed(42)
rng = np.random.default_rng(42)
from scipy.special import factorial
from scipy import stats
import heapq

In [ ]:
def confidence(est,alpha, data, n):
    z = stats.norm.ppf(1 - alpha / 2)
    std = np.std(data, ddof=1)
    error = z * std / np.sqrt(n)
    lower = est - error
    upper = est + error
    print(f"{(1 - alpha) * 100:.1f}% confidence interval: \n[{lower:.4f}, {upper:.4f}]")


## Part 1

In [ ]:
# Monte Carlo estimator
np.random.seed(42)
n_mc = 100
Ui_mc = np.random.uniform(0, 1, n_mc)
Xi = np.exp(Ui_mc)
est_mc = 1/n_mc * np.sum(Xi)
print(f"Monte Carlo estimator: {est_mc:.4f}")

# Konfidens intervaller
alpha_mc = 0.05
confidence(est_mc, alpha_mc, Xi, n_mc)

## Part 2

In [ ]:
# Antithetic variables
np.random.seed(42)
n_av = 100
Ui_av = np.random.uniform(0, 1, n_av)
Xi_av = np.exp(Ui_av)
Yi_av = (np.exp(Ui_av)+np.exp(1 - Ui_av)) / 2
est_av = 1/n_av * np.sum(Yi_av)
print(f"Antithetic variables estimator: {est_av:.4f}")

# Konfidens interval
alpha_av = 0.05
confidence(est_av, alpha_av, Yi_av, n_av)

# Part 3

In [ ]:
# control variates
np.random.seed(42)
n_cv = 100
Ui_cv = np.random.uniform(0, 1, n_cv)
Xi_cv = np.exp(Ui_cv)
Zi = Ui_cv
mu_Z = 0.5
c = -np.cov(Xi_cv, Zi)[0, 1] / np.var(Zi)
Yi_cv = Xi_cv + c * (Zi - mu_Z)
est_cv = 1/n_cv * np.sum(Yi_cv)
print(f"Control variates estimator: {est_cv:.4f}")  

# Konfidens interval
alpha_cv = 0.05
confidence(est_cv, alpha_cv, Yi_cv, n_cv)

## Part 4

In [ ]:
# Stratified sampling
np.random.seed(42)
n_ss = 100
strata = 10
m = n_ss // strata

Uss = np.zeros((m, strata))

for j in range(1, strata+1):
    a = (j-1)/strata
    b = j/strata
    Uss[:, j-1] = np.random.uniform(a, b, m)

Xi_ss = np.exp(Uss)
Yi_ss = 1/strata * np.sum(Xi_ss, axis=1)

est_ss = 1/m * np.sum(Yi_ss)
print(f"Stratified sampling estimator: {est_ss:.4f}")

# Konfidens interval
alpha_ss = 0.05
confidence(est_ss, alpha_ss, Yi_ss, m)

## Part 5

In [ ]:
def sim_block_system(m, num_customers, arrival_gen, service_gen):
    clock = 0.0
    blocked_count = 0
    # priority kø
    servers = [] 
    
    for _ in range(num_customers):
        clock += arrival_gen()
        
        while servers and servers[0] <= clock:
            heapq.heappop(servers)
            
        if len(servers) < m:
            service_time = service_gen()
            heapq.heappush(servers, clock + service_time)
        else:
            blocked_count += 1
            
    return blocked_count / num_customers

In [ ]:
# Control variates for blocking probability
np.random.seed(42)
n_pcv = 100

m = 10
mean_service = 8.0
mean_interarrival = 1.0
num_customers = 10000

arrival_gen = lambda: np.random.exponential(mean_interarrival)

Xi_pcv = np.zeros(n_pcv)
Zi_pcv = np.zeros(n_pcv)  

for i in range(n_pcv):
    total_service = [0.0]  

    def service_gen_pcv():
        s = np.random.exponential(mean_service)
        total_service[0] += s
        return s

    Xi_pcv[i] = sim_block_system(m, num_customers, arrival_gen, service_gen_pcv)

    Zi_pcv[i] = total_service[0]

mu_Z_pcv = np.mean(Zi_pcv)

c_pcv = -np.cov(Xi_pcv, Zi_pcv, ddof=1)[0, 1] / np.var(Zi_pcv, ddof=1)

Yi_pcv = Xi_pcv + c_pcv * (Zi_pcv - mu_Z_pcv)

# estimator
est_pcv = np.mean(Yi_pcv)
print(f"Control variates estimator: {est_pcv:.4f}")

# Konfidens interval
alpha_pcv = 0.05
confidence(est_pcv, alpha_pcv, Yi_pcv, n_pcv)

## Part 6

In [ ]:
np.random.seed(42)

m = 10
mean_service = 8.0
mean_interarrival = 1.0
num_customers = 10000
num_runs = 10

p = 0.8
mean_fast = 1/0.8333 
mean_slow = 1/5.0     

service_gen = lambda: np.random.exponential(mean_service)

# uden CRN
arrival_gen_poisson = lambda: np.random.exponential(mean_interarrival)

def hyperexponential_arrival():
    if np.random.rand() < p:
        return np.random.exponential(mean_fast)
    else:
        return np.random.exponential(mean_slow)


# med CRN 

U1 = np.random.rand(num_customers)  
U2 = np.random.rand(num_customers) 

def poisson_arrivals_CRN():
    i = 0
    def gen():
        nonlocal i
        x = -mean_interarrival * np.log(1 - U1[i])
        i += 1
        return x
    return gen

def hyperexp_arrivals_CRN():
    i = 0
    def gen():
        nonlocal i
        if U1[i] < p:
            x = -mean_fast * np.log(1 - U2[i])
        else:
            x = -mean_slow * np.log(1 - U2[i])
        i += 1
        return x
    return gen


# uden CRN

results_p1 = [
    sim_block_system(m, num_customers, arrival_gen_poisson, service_gen)
    for _ in range(num_runs)
]

results_p2 = [
    sim_block_system(m, num_customers, hyperexponential_arrival, service_gen)
    for _ in range(num_runs)
]

# med CRN
results_p1_CRN = [
    sim_block_system(m, num_customers, poisson_arrivals_CRN(), service_gen)
    for _ in range(num_runs)
]

results_p2_CRN = [
    sim_block_system(m, num_customers, hyperexp_arrivals_CRN(), service_gen)
    for _ in range(num_runs)
]


# varians sammenligning
diff_noCRN = np.array(results_p1) - np.array(results_p2)
diff_CRN   = np.array(results_p1_CRN) - np.array(results_p2_CRN)

alpha = 0.05
print("\n--- UDEN CRN ---")
print("Poisson mean:", np.mean(results_p1))
print("HyperExp mean:", np.mean(results_p2))
print("Var(diff):", np.var(diff_noCRN, ddof=1))

print("\n--- MED CRN ---")
print("Poisson mean:", np.mean(results_p1_CRN))
print("HyperExp mean:", np.mean(results_p2_CRN))
print("Var(diff):", np.var(diff_CRN, ddof=1))

## Part 7

In [ ]:
import numpy as np
from scipy.stats import norm

# Crude MC estimator for Normal fordeling
def crude_mc(a, n):
    Z = np.random.randn(n)
    return np.mean(Z > a)

def importance_sampling(a, n, sigma=1.0):
    Y = np.random.normal(loc=a, scale=sigma, size=n)
    f = norm.pdf(Y, 0, 1)
    g = norm.pdf(Y, a, sigma)
    w = f / g
    return np.mean((Y > a) * w)

for a in [2, 4]:
    for n in [1000, 10000, 100000]:
        crude = crude_mc(a, n)
        est_is = importance_sampling(a, n, sigma=1.0)
        print(f"a={a}, n={n}: crude={crude:.4f}, estimator={est_is:.6f}")

        # Konfidens interval for crude MC
        alpha = 0.05
        print(f"Crude")
        confidence(crude, alpha, (np.random.randn(n) > a).astype(float), n)
        # Konfidens interval for IS estimator
        print(f"Importance Sampling")
        confidence(est_is, alpha, (importance_sampling(a, n, sigma=1.0) * (np.random.normal(loc=a, scale=1.0, size=n) > a)).astype(float), n)

## Part 8

In [ ]:
def h(x):
    return np.exp(x)

def f(x):
    return (0 <= x) & (x <= 1)

def g(x, lam):
    return lam * np.exp(-lam * x)

def theta_IS(lam, Y):
    estimator = h(Y) * f(Y) / g(Y, lam)

    return np.mean(estimator)

lam = 1.0
Y = np.random.exponential(scale=1/lam, size=10000)
print(theta_IS(lam, Y))
confidence(theta_IS(lam, Y), 0.05, (h(Y) * f(Y) / g(Y, lam)).astype(float), len(Y))